In [ ]:
import torch
import numpy as np
import pandas as pd
import time
from PIL import Image
import os
import shutil
import random
import torch
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
import tensorflow as tf
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import CarliniLInfMethod


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

MAK_DATASET_ONE_PER_PERSON = 0
WORKER = "Salvatore"

DATA_DIR_DF_CW = './dataset/dataset_1_per_person'
RESULTS_DIR = './results'
CSV_PATH = './dataset/selected_100_subjects.csv'
BASE_ADVERSARIAL_CW_DIR = './adversarial_data/CW_Untargeted'

print("\n[Fase 1] Caricamento del modello Target NN1 (InceptionResnetV1)...")
nn1 = InceptionResnetV1(pretrained='vggface2').eval().to(device)
nn1.classify = True

# Selecting 1 image for subject

In this section we will select just one image per person. So we will build a subset for CW and DeepFool attack.

In [ ]:
if MAK_DATASET_ONE_PER_PERSON:
    # ==============================================================================
    # Impostazione dei percorsi
    # ==============================================================================
    SOURCE_DIR = './dataset/dataset_100_subjects'
    DEST_DIR = './dataset/dataset_1_per_person'

    # Crea la directory di destinazione (se esiste già, non fa nulla)
    os.makedirs(DEST_DIR, exist_ok=True)

    print("Inizio la selezione di un'immagine per soggetto...")

    # Contatore per tenere traccia delle immagini elaborate
    processed_count = 0

    # ==============================================================================
    # Estrazione e copia
    # ==============================================================================
    # Scansiona tutte le cartelle all'interno del dataset locale
    for subject_folder in os.listdir(SOURCE_DIR):
        subject_path = os.path.join(SOURCE_DIR, subject_folder)

        # Verifica che sia effettivamente una cartella
        if os.path.isdir(subject_path):
            # Filtra solo i file immagine
            images = [f for f in os.listdir(subject_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

            if images:
                selected_image = random.choice(images)

                src_image_path = os.path.join(subject_path, selected_image)

                # Estrae l'estensione originale (.jpg, .png, ecc.)
                ext = os.path.splitext(selected_image)[1]

                # Rinomina il file con l'ID del soggetto per mantenere la tracciabilità assoluta
                new_filename = f"{subject_folder}{ext}"
                dest_image_path = os.path.join(DEST_DIR, new_filename)

                # Copia il file nella nuova destinazione
                shutil.copy2(src_image_path, dest_image_path)
                processed_count += 1

    print(f"\nOperazione completata! Sono state estratte {processed_count} immagini.")
    print(f"Le puoi trovare nella cartella: {DEST_DIR}")

In [ ]:
print("Downloading official VGGFace2 label mapping...")
fpath = tf.keras.utils.get_file(
    'rcmalli_vggface_labels_v2.npy',
    "https://github.com/rcmalli/keras-vggface/releases/download/v2.0/rcmalli_vggface_labels_v2.npy",
    cache_subdir="./"
)
LABELS = np.load(fpath)

print("Lettura del CSV e sincronizzazione degli ID in corso...")
df_subjects = pd.read_csv(CSV_PATH)

df_subjects['Class_ID'] = df_subjects['Class_ID'].astype(str).str.strip()
df_subjects['Name'] = df_subjects['Name'].astype(str).str.strip()

label_to_index = {}

for _, row in df_subjects.iterrows():
    c_id = row['Class_ID']
    p_name = row['Name']

    found_idx = -1
    for idx, lbl in enumerate(LABELS):
        lbl_str = str(lbl).strip().strip("'").strip('"')
        if c_id in lbl_str or p_name in lbl_str:
            found_idx = idx
            break

    if found_idx != -1:
        label_to_index[c_id] = found_idx
    else:
        print(f"ERRORE (Ignorabile): Impossibile trovare '{c_id}' o '{p_name}'")

print(f"Mappatura completata con successo. Trovati {len(label_to_index)} soggetti validi.")

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

# Il range visivo è [0.0, 1.0]. Il vincolo massimo di L_inf è 0.1.
EPSILON_LIMIT = 0.1

print(f"--- AVVIO SCRIPT - LAVORATORE {WORKER} su {device} ---")
print("\n[Fase 2] Preprocessing e Caricamento delle immagini...")

preprocess_attack = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor()
])

x_data = []
y_labels_numeric = []
image_names = []

image_files = sorted([f for f in os.listdir(DATA_DIR_DF_CW) if f.lower().endswith(('.jpg', '.png'))])

if len(image_files) != 100:
    print(f"ATTENZIONE: Trovate {len(image_files)} immagini invece di 100!")

for img_name in image_files:
    img_path = os.path.join(DATA_DIR_DF_CW, img_name)

    img = Image.open(img_path).convert('RGB')
    img_tensor = preprocess_attack(img)
    x_data.append(img_tensor.numpy())
    image_names.append(img_name)

    class_id = os.path.splitext(img_name)[0].strip()

    if class_id in label_to_index:
        y_labels_numeric.append(label_to_index[class_id])
    else:
        print(f"Errore critico: ID {class_id} non mappato.")
        y_labels_numeric.append(0)

x_data = np.array(x_data)
num_classes = len(LABELS)
y_data_one_hot = np.eye(num_classes)[y_labels_numeric]

print(f"Caricate {len(x_data)} immagini. Shape: {x_data.shape}")

# ==============================================================================
# WRAPPER ART E VALUTAZIONE BASELINE
# ==============================================================================
print("\n[Fase 3] Configurazione ART Classifier...")

loss_fn = torch.nn.CrossEntropyLoss()

classifier = PyTorchClassifier(
    model=nn1,
    clip_values=(0.0, 1.0),
    loss=loss_fn,
    optimizer=None,
    input_shape=(3, 160, 160),
    nb_classes=num_classes,
    preprocessing=(np.array([0.5, 0.5, 0.5]), np.array([0.5, 0.5, 0.5])),
    device_type='gpu' if torch.cuda.is_available() else 'cpu'
)

if WORKER == "Christian":
    max_iter_list = [1]
    start_idx, end_idx = 0, 100
elif WORKER == "Salvatore":
    max_iter_list = [5]
    start_idx, end_idx = 0, 100
elif WORKER == "Candido":
    max_iter_list = [10]
    start_idx, end_idx = 0, 100
else:
    raise ValueError("WORKER non valido.")

x_test_subset = x_data[start_idx:end_idx]
y_test_subset = y_data_one_hot[start_idx:end_idx]
image_names_subset = image_names[start_idx:end_idx]

print(f"Lavoratore {WORKER}: assegnate {len(x_test_subset)} immagini.")

# --- CALCOLO BASELINE (PULITA) ---
print("\nCalcolo della baseline sulle immagini originali...")
preds_clean = np.argmax(classifier.predict(x_test_subset), axis=1)
true_labels = np.argmax(y_test_subset, axis=1)
correct_clean_mask = (preds_clean == true_labels)
num_correct_clean = np.sum(correct_clean_mask)
baseline_acc = (num_correct_clean / len(x_test_subset)) * 100

print(f"Accuratezza Baseline (Clean): {num_correct_clean}/{len(x_test_subset)} ({baseline_acc:.2f}%) classificate correttamente.")

# ==============================================================================
# GRID SEARCH CW L_INF (CON FILTRO L_INF > 0.10)
# ==============================================================================
print("\n[Fase 4] Avvio attacco C&W L_inf (con scarto per perturbazioni > 0.10)...")

learning_rate_list = [0.001, 0.01, 0.05, 0.1]
confidence_list = [0.0, 0.5, 1.0]

results = []
total_combinations = len(max_iter_list) * len(learning_rate_list) * len(confidence_list)
current_combo = 1

for m_iter in max_iter_list:
    for lr in learning_rate_list:
        for conf in confidence_list:
            print(f"\n[{current_combo}/{total_combinations}] Configurazione: max_iter={m_iter}, lr={lr}, confidence={conf}")
            start_time_combo = time.time()

            # Creazione della stringa del percorso
            lr_str = str(lr).replace('.', '_')
            conf_str = str(conf).replace('.', '_')
            combo_dir_name = f"max_iter_{m_iter}_lr_{lr_str}_c_{conf_str}"

            output_images_dir = os.path.join(BASE_ADVERSARIAL_DIR, combo_dir_name)
            os.makedirs(output_images_dir, exist_ok=True)

            attack_cw = CarliniLInfMethod(
                classifier=classifier,
                targeted=False,
                max_iter=m_iter,
                learning_rate=lr,
                confidence=conf,
                verbose=False
            )

            x_adv_raw_list = []
            print(f"   -> Generazione in corso per {len(x_test_subset)} immagini...")

            # --- Generazione attacchi con misurazione dei tempi ---
            for i in range(len(x_test_subset)):
                start_img_time = time.time()

                single_adv = attack_cw.generate(x=x_test_subset[i:i+1])

                img_time = time.time() - start_img_time
                print(f"      - Immagine {i+1:02d}/{len(x_test_subset)} ({image_names_subset[i]}) processata in {img_time:.2f} secondi")

                x_adv_raw_list.append(single_adv)

            x_adv_raw = np.concatenate(x_adv_raw_list, axis=0)

            # Calcoliamo la perturbazione originale senza applicare il clipping forzato
            perturbation_raw = x_adv_raw - x_test_subset

            l_inf_applied_list = []
            saved_count = 0

            # --- CONTROLLO E SALVATAGGIO SINGOLE IMMAGINI ---
            for i in range(len(x_adv_raw)):
                # Calcolo della perturbazione L_inf specifica per l'i-esima immagine
                l_inf_img = np.max(np.abs(perturbation_raw[i]))

                # Aggiungiamo 1e-5 per evitare che errori di precisione nei float (es. 0.1000001) scartino immagini valide
                if l_inf_img > EPSILON_LIMIT + 1e-5:
                    print(f"      - Immagine {image_names_subset[i]}: per questa determinata config l'algoritmo CW è stato in grado di trovare solo una perturbazione superiore a 0.10 (L_inf = {l_inf_img:.4f}) tale che ingannasse la rete e quindi non è stata salvata nella cartella della config corrente.")
                    continue

                # Se passa il controllo, limitiamo i pixel al range visivo [0.0, 1.0] per salvarla
                adv_img_np = np.clip(x_adv_raw[i], 0.0, 1.0)

                # Conversione per il salvataggio PIL
                adv_img_np_save = np.transpose(adv_img_np, (1, 2, 0))
                adv_img_np_save = (adv_img_np_save * 255.0).astype(np.uint8)
                adv_pillow_img = Image.fromarray(adv_img_np_save)

                # Salvataggio
                img_save_path = os.path.join(output_images_dir, image_names_subset[i])
                adv_pillow_img.save(img_save_path)

                saved_count += 1

                # Calcoliamo la perturbazione effettiva impressa sulla foto (dopo il clipping visivo in [0, 1])
                actual_perturbation = adv_img_np - x_test_subset[i]
                l_inf_applied_list.append(np.max(np.abs(actual_perturbation)))

            # Media della perturbazione per questa configurazione (solo sulle immagini valide salvate)
            if saved_count > 0:
                mean_linf_applied = np.mean(l_inf_applied_list)
            else:
                mean_linf_applied = 0.0

            elapsed_time_combo = time.time() - start_time_combo

            # Log richiesto nel CSV
            results.append({
                'WORKER': WORKER,
                'max_iter': m_iter,
                'learning_rate': lr,
                'confidence': conf,
                'Mean_Linf_Applied': round(mean_linf_applied, 4),
                'Time_Elapsed (s)': round(elapsed_time_combo, 2)
            })

            print(f" --> RISULTATO: Salvate {saved_count}/{len(x_test_subset)} immagini | L_inf media applicata: {mean_linf_applied:.4f} | Tempo Totale Configurazione: {elapsed_time_combo:.1f}s")
            current_combo += 1

# ==============================================================================
# FASE 5: SALVATAGGIO DEI DATI DEL LAVORATORE
# ==============================================================================
df_results = pd.DataFrame(results)
csv_filename = f'cw_grid_search_worker_{WORKER}.csv'
csv_filepath = os.path.join(RESULTS_DIR, csv_filename)

df_results.to_csv(csv_filepath, index=False)

print("\n=======================================================")
print(f"Lavoro completato! Risultati salvati in:\n{csv_filepath}")
print("=======================================================")